In [ ]:
import torch
import os
import librosa
import numpy as np
import soundfile as sf
from scipy.io import wavfile
from IPython.display import Audio
from torch.nn.utils.rnn import pad_sequence
from transformers import Wav2Vec2Model, Wav2Vec2Tokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor 

In [ ]:
def wav2vec(file_name, extractor):
    print(file_name)
    data = wavfile.read(file_name)
    framerate = data[0]
    sounddata = data[1]
    time = np.arange(0,len(sounddata))/framerate
    input_audio, _ = librosa.load(file_name, sr=16000)
    input_values = extractor(input_audio, return_tensors="pt", sampling_rate=16000, model_max_length = 1000).input_values
    audio_vector = input_values[0]
    """    print(type(input_values))
    vectorLength = input_values.shape[1]
    
    audio_vector = torch.reshape(input_values, (vectorLength))"""
    """audio_vector = input_values.resize_(800,)
    audio_vector = audio_vector.numpy()
    return audio_vector"""
    return audio_vector

def text_save(filename, data):#filename为写入txt文件的路径，data为要写入数据列表.
    file = open(filename,'a')
    for i in range(len(data)):
        s = str(data[i]).replace('[','').replace(']','')#去除[],这两行按数据不同，可以选择
        s = s.replace("'",'').replace(',','') +'\n'   #去除单引号，逗号，每行末尾追加换行符
        file.write(s)
    file.close()
    print("保存文件成功") 
    
def read_txt_to_list(file_path):  
    result=[]
    file = open(file_path)
    contents = file.read().splitlines()
    return contents

In [ ]:
# 数据集地址
DATASET_DIR = os.environ.get("FAKEAVCELEB_DIR", "data/FakeAVCeleb")
listType = os.listdir(DATASET_DIR)  # 列出文件夹下所有的目录与文件

# 预训练模型地址
WAV2VEC_MODEL_DIR = "wav2vec2-base-960"
extractor = Wav2Vec2FeatureExtractor.from_pretrained(WAV2VEC_MODEL_DIR)

# 模型输出地址
output_dir = "wav2vec-finetune"

In [ ]:
# 示例用法
file_name = 'sa1_1.wav'

audioVector = wav2vec(file_name, extractor)



In [ ]:
audioVector

In [ ]:
# 存放音频向量
audioVecX = []

In [ ]:
for i in range(0, len(listType)):
    pathType = os.path.join(DATASET_DIR, listType[i])
    # e.g: ..\FakeAVCeleb_v1.2\FakeVideo-FakeAudio\
    listLocation = os.listdir(pathType)
    for j in range(0, len(listLocation)):
        pathLocation = os.path.join(pathType, listLocation[j])
        # e.g:  ..\FakeAVCeleb_v1.2\FakeVideo-FakeAudio\African\
        listGender = os.listdir(pathLocation)
        for k in range(0, len(listGender)):
            pathGender = os.path.join(pathLocation, listGender[k])
            # e.g: ..\FakeAVCeleb_v1.2\FakeVideo-FakeAudio\African\men\
            listId = os.listdir(pathGender)
            for l in range(0, len(listId)):
                pathId = os.path.join(pathGender, listId[l])
                # e.g: ..\FakeAVCeleb_v1.2\FakeVideo-FakeAudio\African\men\id001166
                listVideo = os.listdir(pathId)
                
                for file in listVideo:
                    pathFile = os.path.join(pathId, file)
                    # 判断文件类型
                    ext = os.path.splitext(pathFile)[1]
                    if ext ==".wav":
                        audio_vector = wav2vec(pathFile, extractor)
                        audioVecX.append(audio_vector)

In [ ]:


audioVecX = pad_sequence(audioVecX)
audioVecX.shape

In [ ]:
print(abc.numpy().T.shape)
audioVecX = np.array(audioVecX)
audioVecX.shape

In [ ]:
%reset

## delete

In [ ]:
train_data = []
for i in range(0, len(listType)):
    pathType = os.path.join(DATASET_DIR, "RealVideo-RealAudio")
    # e.g: ..\FakeAVCeleb_v1.2\FakeVideo-FakeAudio\
    listLocation = os.listdir(pathType)
    for j in range(0, len(listLocation)):
        pathLocation = os.path.join(pathType, listLocation[j])
        # e.g:  ..\FakeAVCeleb_v1.2\FakeVideo-FakeAudio\African\
        listGender = os.listdir(pathLocation)
        for k in range(0, len(listGender)):
            pathGender = os.path.join(pathLocation, listGender[k])
            # e.g: ..\FakeAVCeleb_v1.2\FakeVideo-FakeAudio\African\men\
            listId = os.listdir(pathGender)
            for l in range(0, len(listId)):
                pathId = os.path.join(pathGender, listId[l])
                # e.g: ..\FakeAVCeleb_v1.2\FakeVideo-FakeAudio\African\men\id001166
                listVideo = os.listdir(pathId)
                
                info = {}
                for file in listVideo:
                    pathFile = os.path.join(pathId, file)
                    # 判断文件类型
                    ext = os.path.splitext(pathFile)[1]
                    if ext ==".wav":
                        info['path'] = pathFile
                    if ext ==".txt":
                        file = open(pathFile, 'r')
                        sentence = file.read().lower()
                        info['transcription'] = sentence
                        #print(sentence)
                        file.close()
                train_data.append(info)

In [ ]:
text_save("train_data.txt", train_data)#filename为写入txt文件的路径，data为要写入数据列表.


In [ ]:
len(train_data)

In [ ]:
"""!pip install huggingsound"""

In [ ]:
from huggingsound import TrainingArguments, ModelArguments, SpeechRecognitionModel, TokenSet

model = SpeechRecognitionModel(WAV2VEC_MODEL_DIR)
output_dir = "wav2vec2-finetune"

# first of all, you need to define your model's token set
# however, the token set is only needed for non-finetuned models
# if you pass a new token set for an already finetuned model, it'll be ignored during training
tokens = ["a", "b", "c", "d", "e", "f", "g", "h", "i", "j", "k", "l", "m", "n", "o", "p", "q", "r", "s", "t", "u", "v", "w", "x", "y", "z", "'"]
token_set = TokenSet(tokens)




In [ ]:
""" = [
    {"path": "/path/to/sagan.mp3", "transcription": "extraordinary claims require extraordinary evidence"},
    {"path": "/path/to/asimov.wav", "transcription": "violence is the last refuge of the incompetent"},
]"""
# define your custom train data
# train_data = read_txt_to_list("train_data.txt")


In [ ]:
"""!pip uninstall -y transformers accelerate
!pip install transformers accelerate"""

In [ ]:
train_data

In [ ]:
# and finally, fine-tune your model
model.finetune(
    output_dir, 
    train_data=train_data,
    token_set=token_set,
)

In [ ]:
WAV2VECFINETUNE_MODEL_DIR = "wav2vec2-finetune"
extractor = Wav2Vec2FeatureExtractor.from_pretrained(WAV2VECFINETUNE_MODEL_DIR)